In [ ]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
path = "/kaggle/input/datasets/nolasthitnotomorrow/radioml2016-deepsigcom/RML2016.10a_dict.pkl"

with open(path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

X, y, snr_list = [], [], []

for (mod, snr), samples in data.items():
    for sample in samples:
        X.append(sample)
        y.append(mod)
        snr_list.append(snr)

X = np.array(X)
y = np.array(y)
snr_list = np.array(snr_list)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(np.unique(y_encoded))

# Reshape → (N,1,2,128)
X = X.reshape(-1, 1, 2, 128)

print("Dataset shape:", X.shape)
print("Number of classes:", num_classes)

In [ ]:
X_train, X_temp, y_train, y_temp, snr_train, snr_temp = train_test_split(
    X, y_encoded, snr_list, test_size=0.4, stratify=y_encoded, random_state=42
)

X_val, X_test, y_val, y_test, snr_val, snr_test = train_test_split(
    X_temp, y_temp, snr_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=1024)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)

In [ ]:
class TDRNN(nn.Module):
    def __init__(self, num_classes=11, gru_hidden=64):
        super(TDRNN, self).__init__()

        self.bn = nn.BatchNorm2d(1)

        self.conv1 = nn.Conv2d(1, 16, kernel_size=(1,3), padding=(0,1))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(2,3), padding=(0,1))
        self.conv3 = nn.Conv2d(32, 64, kernel_size=(1,3), padding=(0,0))

        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(16, 16)
        self.bn_fc = nn.BatchNorm1d(16)
        self.fc2 = nn.Linear(16, 16)

        self.gru = nn.GRU(input_size=64, hidden_size=gru_hidden, batch_first=True)
        self.fc = nn.Linear(gru_hidden, num_classes)

    def threshold_block(self, x):
        abs_x = torch.abs(x)

        beta = self.gap(abs_x)
        beta = beta.squeeze(-1).squeeze(-1)

        beta = F.relu(self.fc1(beta))
        beta = self.bn_fc(beta)
        beta = torch.sigmoid(self.fc2(beta))

        beta = beta.unsqueeze(2).unsqueeze(3)
        tau = 2 * beta

        return torch.sign(x) * F.relu(abs_x - tau)

    def forward(self, x):
        x = self.bn(x)

        x = F.relu(self.conv1(x))
        x = self.threshold_block(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.squeeze(2)
        x = x.permute(0, 2, 1)

        _, h = self.gru(x)

        return self.fc(h[-1])

In [ ]:
def train_model(model, optimizer, epochs=30, save_path="tdrnn_main.pth"):
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    patience = 10
    counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()

        train_acc = 100 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)

                val_loss += criterion(outputs, labels).item()
                _, pred = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (pred == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        print(f"Epoch {epoch+1}: Train={train_acc:.2f}%, Val={val_acc:.2f}%")

        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping")
                break

In [ ]:
def snr_accuracy(model):
    model.eval()
    snr_correct, snr_total = {}, {}

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, pred = torch.max(outputs, 1)

            batch_snrs = snr_test[i*1024:i*1024+inputs.size(0)]

            for j in range(len(labels)):
                snr = batch_snrs[j]
                snr_total[snr] = snr_total.get(snr, 0) + 1
                if pred[j] == labels[j]:
                    snr_correct[snr] = snr_correct.get(snr, 0) + 1

    snrs = sorted(snr_total.keys())
    acc = [100 * snr_correct.get(s,0)/snr_total[s] for s in snrs]

    return snrs, acc

In [ ]:
model = TDRNN(num_classes, 64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_model(model, optimizer)

model.load_state_dict(torch.load("tdrnn_main.pth"))

In [ ]:
snrs, acc = snr_accuracy(model)

# Print SNR-wise accuracy
for s, a in zip(snrs, acc):
    print(f"SNR {s}: {a:.2f}%")

# Plot
plt.figure(figsize=(10,6))
plt.plot(snrs, acc, marker='o')
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("TDRNN Accuracy vs SNR")
plt.grid(True)

# ✅ SAVE PLOT (IMPORTANT FOR KAGGLE)
plt.savefig("snr_plot.png")

plt.show()

In [ ]:
class TDRNN_NoDenoise(TDRNN):
    def forward(self, x):
        x = self.bn(x)

        x = F.relu(self.conv1(x))
        # ❌ skip threshold_block

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.squeeze(2)
        x = x.permute(0, 2, 1)

        _, h = self.gru(x)

        return self.fc(h[-1])

In [ ]:
model_no = TDRNN_NoDenoise(num_classes, 64).to(device)
optimizer_no = torch.optim.Adam(model_no.parameters(), lr=0.001)

train_model(model_no, optimizer_no, epochs=30, save_path="tdrnn_no_denoise.pth")

model_no.load_state_dict(torch.load("tdrnn_no_denoise.pth"))

In [ ]:
snr1, acc1 = snr_accuracy(model)      # WITH denoiser
snr2, acc2 = snr_accuracy(model_no)   # WITHOUT denoiser

In [ ]:
snr1_list = list(snr1)
snr2_list = list(snr2)
acc1_list = list(acc1)
acc2_list = list(acc2)

avg1 = sum(acc1_list) / len(acc1_list)
avg2 = sum(acc2_list) / len(acc2_list)

plt.figure(figsize=(10,6))

# WITH denoiser (green circles)
plt.plot(snr1_list, acc1_list, marker='o', color='green', linewidth=2,
         label=f"TDRNN model with Denoiser [{avg1:.1f}]")

# WITHOUT denoiser (black triangles)
plt.plot(snr2_list, acc2_list, marker='^', color='black', linewidth=2,
         label=f"TDRNN model without Denoiser [{avg2:.1f}]")

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Comparing TDRNN model with and without Threshold Denoiser")

plt.grid(True)
plt.legend()

# ✅ SAVE (VERY IMPORTANT)
plt.savefig("fig6_comparison.png")

plt.show()

In [ ]:
gru_sizes = [16, 32, 64, 128, 256]

In [ ]:
gru_results = {}

for g in gru_sizes:
    print(f"\n🔹 Training GRU {g}")

    model_g = TDRNN(num_classes, gru_hidden=g).to(device)
    optimizer_g = torch.optim.Adam(model_g.parameters(), lr=0.001)

    train_model(model_g, optimizer_g, epochs=60, save_path=f"tdrnn_gru{g}.pth")

    model_g.load_state_dict(torch.load(f"tdrnn_gru{g}.pth"))

    snrs, acc = snr_accuracy(model_g)

    gru_results[g] = (snrs, acc)

In [ ]:
plt.figure(figsize=(10,6))

markers = ['o', 's', '>', 'o', '*']
colors = ['green', 'red', 'blue', 'black', 'goldenrod']

for i, g in enumerate(gru_sizes):
    snrs, acc = gru_results[g]
    
    avg_acc = sum(acc) / len(acc)

    plt.plot(
        snrs,
        acc,
        marker=markers[i],
        color=colors[i],
        linewidth=2,
        label=f"GRU {g} [{avg_acc:.1f}%]"
    )

plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Classification performance of different GRU hidden layers")

plt.grid(True)
plt.legend()

# ✅ SAVE
plt.savefig("fig4_gru_comparison.png")

plt.show()

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def measure_time(model):
    model.eval()
    start = time.time()
    total = 0

    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            model(inputs)
            total += inputs.size(0)

    return (time.time() - start) / total * 1000  # ms/sample

In [ ]:
params = []
times = []

for g in gru_sizes:
    model_g = TDRNN(num_classes, gru_hidden=g).to(device)

    p = count_parameters(model_g)
    t = measure_time(model_g)

    print(f"GRU {g}: Params={p}, Time={t:.4f} ms")

    params.append(p)
    times.append(t)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,6))

# Bar chart → Parameters
ax1.bar([str(g) for g in gru_sizes], params, color='steelblue', alpha=0.7)
ax1.set_xlabel("GRU Hidden Size")
ax1.set_ylabel("Number of Parameters", color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Line → Inference time
ax2 = ax1.twinx()
ax2.plot([str(g) for g in gru_sizes], times, color='orange', marker='o', linewidth=2)
ax2.set_ylabel("Inference Time (ms)", color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

plt.title("Model Complexity vs GRU Hidden Size")

fig.tight_layout()

# ✅ SAVE
plt.savefig("fig5_complexity.png")

plt.show()